# NBA points scoring discovery

Explore what drives `pts`, `pts_per_min`, and `minutes` using existing training parquets + optional `nba_api` probes.

Spec: `docs/superpowers/specs/2026-07-26-nba-pts-scoring-discovery-design.md`

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve repo root (directory that contains data/)
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    for cand in [ROOT.parent, *ROOT.parents]:
        if (cand / "data").exists():
            ROOT = cand
            break
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)
print("cwd:", Path.cwd())

In [ ]:
from src.pipeline.features.context_features import ContextFeatureEngineer
from models.shared.scoring_discovery import (
    build_coverage_map,
    derive_pts,
    lineage_for,
    merge_driver_shortlist,
    rank_univariate,
    season_rank_stability,
    split_feature_pools,
)

SEASONS = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25", "2025-26"]
HOLDOUT_SEASON = "2025-26"
TARGETS = ["pts", "pts_per_min", "minutes"]
RANDOM_SEED = 42
RUN_ENDPOINT_PROBES = False  # set True only when live nba_api pulls are desired
np.random.seed(RANDOM_SEED)

In [ ]:
frames = []
missing = []
for yr in SEASONS:
    path = Path(f"data/processed/{yr}_Regular_Season_training_data.parquet")
    if not path.exists():
        missing.append(yr)
        print(f"⚠ missing parquet for {yr}: {path}")
        continue
    season_df = pd.read_parquet(path)
    season_df = ContextFeatureEngineer(league="nba", season=yr).enrich(season_df)
    frames.append(season_df)
    print(f"✓ {yr}: {len(season_df):,} rows")

if not frames:
    raise FileNotFoundError("No season parquets found under data/processed/")

df = pd.concat(frames, ignore_index=True)
df = derive_pts(df)
df = df[(df["minutes"] >= 5) | (df["starting"] == 1)].copy()
print(f"Combined: {len(df):,} rows × {df.shape[1]} cols | missing seasons: {missing or 'none'}")
df[["season_year", "pts", "pts_per_min", "minutes"]].describe()

In [ ]:
dupes = df.duplicated(subset=["game_id", "player_id"]).sum()
print(f"Duplicate game_id+player_id: {dupes}")
print(df.groupby("season_year").size())
print("Target nulls:", {t: int(df[t].isna().sum()) for t in TARGETS})

## Wishlist coverage

Status is based on columns present **after** load+enrich — not aspirational names.
`partial` = proxy only (e.g. contested FGA ≈ contest rate).

In [ ]:
coverage_df = build_coverage_map(df.columns)
display(coverage_df)
print(coverage_df["status"].value_counts())

## Scoring anatomy (same-game)

**NOT model features.** These are contemporaneous associations that explain how points are produced in the same game. Using them pre-tip is leakage.

In [ ]:
pools = split_feature_pools(df.columns, targets=TARGETS)
print({k: len(v) for k, v in pools.items()})
# leakage audit
overlap = set(pools["same_game"]) & set(pools["predictive"])
print("same_game ∩ predictive:", overlap or "∅ (ok)")

anatomy_ranks = {}
for target in TARGETS:
    ranks = rank_univariate(df, pools["same_game"], target, random_state=RANDOM_SEED)
    anatomy_ranks[target] = ranks
    print(f"\n=== same-game drivers of {target} ===")
    display(ranks.head(20))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pairs = [
    ("fga_per_min", "pts"),
    ("tchs_per_min", "pts_per_min"),
    ("minutes", "pts"),
]
sample = df.sample(min(8000, len(df)), random_state=RANDOM_SEED)
for ax, (x, y) in zip(axes, pairs):
    if x not in sample.columns or y not in sample.columns:
        ax.set_title(f"missing {x}/{y}")
        continue
    ax.scatter(sample[x], sample[y], s=4, alpha=0.15)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
plt.tight_layout()
plt.show()

## Predictive discovery (prior-only, leak-safe)

Candidates are EWM / season_avg / lag / roll / team / opp / context / market columns only.
Model-based ranks fit on seasons **before** `HOLDOUT_SEASON`; holdout MAE is diagnostic.

In [ ]:
pred_cols = pools["predictive"]
# Drop any accidental target leakage in feature names
pred_cols = [c for c in pred_cols if c not in TARGETS]

pred_ranks = {}
stability = {}
for target in TARGETS:
    r = rank_univariate(df, pred_cols, target, random_state=RANDOM_SEED)
    pred_ranks[target] = r
    stability[target] = season_rank_stability(
        df, pred_cols, target, season_col="season_year", top_n=40, random_state=RANDOM_SEED,
    )
    print(f"\n=== prior drivers of {target} ===")
    display(r.head(25))
    print(f"--- stability ({target}) ---")
    display(stability[target].head(15))

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False
    print("shap not available — skipping SHAP tables")

train_mask = df["season_year"] != HOLDOUT_SEASON
hold_mask = df["season_year"] == HOLDOUT_SEASON
train_df = df.loc[train_mask]
hold_df = df.loc[hold_mask]

XGB_DISC = dict(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    n_jobs=4,
)

shap_tables = {}
perm_tables = {}
holdout_diag = {}

for target in TARGETS:
    top_feats = [
        f for f in pred_ranks[target]["feature"].head(40).tolist()
        if f in train_df.columns
    ]
    X_tr = train_df[top_feats].apply(pd.to_numeric, errors="coerce")
    y_tr = train_df[target].astype(float)
    med = X_tr.median()
    X_tr = X_tr.fillna(med)
    model = XGBRegressor(**XGB_DISC)
    model.fit(X_tr, y_tr)

    if len(hold_df):
        X_ho = hold_df[top_feats].apply(pd.to_numeric, errors="coerce").fillna(med)
        y_ho = hold_df[target].astype(float)
        pred = model.predict(X_ho)
        if target == "pts_per_min" and "base_pts_per_min_season_avg" in hold_df.columns:
            holdout_diag[target] = {
                "model_mae": float(mean_absolute_error(y_ho, pred)),
                "naive_mae": float(mean_absolute_error(
                    y_ho, hold_df["base_pts_per_min_season_avg"].astype(float),
                )),
                "naive": "base_pts_per_min_season_avg",
            }
        elif target == "minutes" and "base_min_season_avg" in hold_df.columns:
            holdout_diag[target] = {
                "model_mae": float(mean_absolute_error(y_ho, pred)),
                "naive_mae": float(mean_absolute_error(
                    y_ho, hold_df["base_min_season_avg"].astype(float),
                )),
                "naive": "base_min_season_avg",
            }
        elif (
            target == "pts"
            and "base_pts_per_min_season_avg" in hold_df.columns
            and "track_minutes_season_avg" in hold_df.columns
        ):
            naive = (
                hold_df["base_pts_per_min_season_avg"].astype(float)
                * hold_df["track_minutes_season_avg"].astype(float)
            )
            holdout_diag[target] = {
                "model_mae": float(mean_absolute_error(y_ho, pred)),
                "naive_mae": float(mean_absolute_error(
                    y_ho, naive.fillna(y_ho.median()),
                )),
                "naive": "ppm_season_avg * track_minutes_season_avg",
            }
        else:
            holdout_diag[target] = {
                "model_mae": float(mean_absolute_error(y_ho, pred)),
                "naive_mae": None,
                "naive": None,
            }

    sample_idx = X_tr.sample(min(5000, len(X_tr)), random_state=RANDOM_SEED).index
    perm = permutation_importance(
        model,
        X_tr.loc[sample_idx],
        y_tr.loc[sample_idx],
        n_repeats=5,
        random_state=RANDOM_SEED,
        n_jobs=2,
    )
    perm_tables[target] = pd.DataFrame({
        "feature": top_feats,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }).sort_values("importance_mean", ascending=False)

    if HAS_SHAP:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_tr.loc[sample_idx])
        shap_tables[target] = pd.DataFrame({
            "feature": top_feats,
            "mean_abs_shap": np.abs(sv).mean(axis=0),
        }).sort_values("mean_abs_shap", ascending=False)
    else:
        shap_tables[target] = pd.DataFrame(columns=["feature", "mean_abs_shap"])

    print(f"\n=== model ranks {target} ===")
    display(perm_tables[target].head(15))
    if HAS_SHAP:
        display(shap_tables[target].head(15))

print("Holdout diagnostic:", holdout_diag)

In [ ]:
from models.shared.analysis import analyze_correlations

for target in TARGETS:
    feats = pred_ranks[target]["feature"].head(20).tolist()
    analyze_correlations(train_df, feats, threshold=0.95, title=f"Predictive corr ({target})")

## Endpoint probes (optional)

Inventory-only pulls for surfaces not in `GameLogs`. No DB upserts.
Default `RUN_ENDPOINT_PROBES = False`.

In [ ]:
from datetime import datetime
import time
import json

PROBE_CACHE = Path("data/raw/cache/discovery")
PROBE_CACHE.mkdir(parents=True, exist_ok=True)

probe_inventory = pd.DataFrame(columns=["endpoint", "ok", "n_rows", "n_cols", "columns_head", "error"])

def _probe(name: str, fn):
    global probe_inventory
    cache_path = PROBE_CACHE / f"{name}.parquet"
    try:
        time.sleep(0.6)
        df_p = fn()
        if df_p is None or df_p.empty:
            raise RuntimeError("empty frame")
        df_p.to_parquet(cache_path, index=False)
        row = {
            "endpoint": name,
            "ok": True,
            "n_rows": len(df_p),
            "n_cols": df_p.shape[1],
            "columns_head": ", ".join(map(str, df_p.columns[:20])),
            "error": "",
        }
    except Exception as exc:
        row = {
            "endpoint": name,
            "ok": False,
            "n_rows": 0,
            "n_cols": 0,
            "columns_head": "",
            "error": str(exc)[:200],
        }
    probe_inventory = pd.concat([probe_inventory, pd.DataFrame([row])], ignore_index=True)
    print(row)

if RUN_ENDPOINT_PROBES:
    from nba_api.stats.endpoints import (
        leaguehustlestatsplayer,
        leaguedashplayerptshot,
        leaguedashptdefend,
        boxscorematchupsv3,
        playergamelogs,
    )

    season = "2024-25"

    _probe(
        "LeagueHustleStatsPlayer",
        lambda: leaguehustlestatsplayer.LeagueHustleStatsPlayer(
            season=season, per_mode_time="PerGame",
        ).get_data_frames()[0],
    )
    _probe(
        "LeagueDashPlayerPtShot",
        lambda: leaguedashplayerptshot.LeagueDashPlayerPtShot(
            season=season, per_mode_simple="PerGame",
        ).get_data_frames()[0],
    )
    _probe(
        "LeagueDashPtDefend",
        lambda: leaguedashptdefend.LeagueDashPtDefend(
            season=season, defense_category="Overall",
        ).get_data_frames()[0],
    )
    # Extra measure types on PlayerGameLogs (sample)
    for measure in ("Misc", "Scoring", "Usage"):
        _probe(
            f"PlayerGameLogs_{measure}",
            lambda m=measure: playergamelogs.PlayerGameLogs(
                season_nullable=season,
                season_type_nullable="Regular Season",
                measure_type_player_game_logs_nullable=m,
            ).get_data_frames()[0].head(500),
        )
    # Matchups: one sample game_id from df if available
    sample_gid = str(df["game_id"].dropna().astype(str).iloc[0]).zfill(10)
    _probe(
        "BoxScoreMatchupsV3",
        lambda: boxscorematchupsv3.BoxScoreMatchupsV3(game_id=sample_gid).get_data_frames()[0],
    )
else:
    print("Skipping endpoint probes (RUN_ENDPOINT_PROBES=False)")

display(probe_inventory)